# AML_C2 — API REST, livraison et suivi MLOps

Conception pédagogique : Mathieu Weill.

**Question :** Comment rendre une prédiction appelable, vérifiable et maintenable ?

[PDF](cours.pdf) · [PowerPoint](presentation.pptx) · [Guide](README.md)

Conserver la structure du kit. Exécuter dans l’ordre depuis un noyau neuf. Ces exemples sont distincts des compétitions Kaggle.

## AML_C2-D1 — Tester le contrat REST

Annoncer une sortie attendue avant de lancer. Les assertions contrôlent le petit exemple ; elles ne prouvent pas une performance universelle.

In [ ]:
from pathlib import Path
import sys
lab=Path.cwd().parent/'laboratoire'
assert (lab/'api.py').exists(), 'Conserver les dossiers du kit ensemble.'
sys.path.insert(0,str(lab.resolve()))
from api import app,train_artifact,drift_report
from fastapi.testclient import TestClient
train_artifact()
row=dict(sepal_length=5.1,sepal_width=3.5,petal_length=1.4,petal_width=.2)
with TestClient(app) as client:
 response=client.post('/predict',json=row)
 print(response.status_code,response.json())
 assert response.status_code==200
 assert client.post('/predict',json={}).status_code==422

## AML_C2-D2 — Distinguer alerte et performance

Annoncer une sortie attendue avant de lancer. Les assertions contrôlent le petit exemple ; elles ne prouvent pas une performance universelle.

In [ ]:
import numpy as np
ref=np.array([[0.,1.],[1.,2.],[2.,3.]])
print('Même distribution :',drift_report(ref,ref)['alert'])
print('Décalage :',drift_report(ref,ref+2)['alert'])
assert not drift_report(ref,ref)['alert']
assert drift_report(ref,ref+2)['alert']

## Mise en service locale réelle

Dans un terminal ouvert dans `laboratoire`, exécuter :

```bash
python api.py
python -m pytest -q
python -m uvicorn api:app --host 127.0.0.1 --port 8000
```

Garder le serveur ouvert. Dans le navigateur, ouvrir **http://127.0.0.1:8000/docs**. Déplier POST /predict, cliquer Try it out puis envoyer les quatre mesures de D1. Vérifier la réponse et le code 200. Retirer un champ : observer le refus 422. Arrêter le serveur avec Ctrl+C.

La démonstration du notebook emploie TestClient sans réseau ; cette étape vérifie un vrai serveur local. Le pipeline reste celui sauvegardé. Ne charger aucun fichier joblib inconnu.

## Mise en pratique CI et maintenance

Copier le contenu de `laboratoire` à la racine d’un dépôt d’exercice personnel, y compris `.github/workflows/ci.yml`. Au prochain push, observer la tâche de tests dans Actions. Dans une branche, modifier une assertion pour la rendre fausse, observer l’échec, puis la corriger. Le service ne se déploie pas automatiquement. La CI du dépôt de cours contrôle aussi le laboratoire.

Conserver `runs/model.joblib` avec son environnement et sa version. Comparer les contrôles avant une nouvelle version. Tenir une fiche : indicateur, seuil, fréquence, responsable, action et version de retour. Le journal local mesure version, classe et latence ; il ne constitue pas une plateforme de supervision de production.


## AML_C2-E — Exercice

La latence double, les entrées changent et les labels ne sont pas encore connus. Que peut-on affirmer ? Quelles actions mener avant de remplacer le modèle ?

Votre réponse et votre justification : …

## Bilan

Appeler /predict, refuser une entrée invalide, tester le rechargement et expliquer une alerte de dérive.

Une sortie expliquée : …

Une limite : …

Une prochaine vérification : …